In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import tifffile
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.predict import load_predict_config, load_predictor

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
AXES = ("axis 0", "axis 1", "axis 2")

In [ ]:
cfg = load_predict_config(PREDICT_CONFIG)
run = ROOT / cfg.run_dir
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pred = load_predictor(run, device=dev)
opts = cfg.make_options(pred)
if opts.volume_size == pred.image_size:
    raise ValueError("Set scale.enabled to true in config/predict.yaml.")

start = time.perf_counter()
vol, stats = pred.predict(opts)
secs = time.perf_counter() - start

out = run / "predictions" / f"mpdd_{opts.volume_size}.tif"
out.parent.mkdir(parents=True, exist_ok=True)
tifffile.imwrite(out, vol.numpy(), photometric="minisblack")

fracs = [round(v, 4) for v in stats["phase_fractions"]]
print(f"shape={tuple(vol.shape)} elapsed={secs:.1f}s")
print(f"fractions={fracs} tiff={out}")

In [ ]:
idx = torch.linspace(0, opts.volume_size - 1, 5).round().int().tolist()
fig, axs = plt.subplots(3, 5, figsize=(15, 9))
for dim, row in enumerate(axs):
    for i, ax in zip(idx, row):
        ax.imshow(
            vol.select(dim, i),
            cmap="gray",
            vmin=0,
            vmax=opts.num_phases - 1,
            interpolation="nearest",
        )
        ax.set_title(f"{AXES[dim]} · slice {i}", fontsize=12)
        ax.axis("off")
plt.tight_layout()

In [ ]:
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vol.numpy(), name="MPDD volume")
viewer